# 23 - Critical Stations Across All Lenses

Every notebook in this project has answered the question *"which stations are critical?"* - and every one of them answered it differently. Notebook 03 said a critical station is a **cut vertex**. Notebook 04 said it is a station with high **betweenness** or high **service volume**. Notebook 05 said it is a station with **no walking alternative** nearby. The extension notebooks add more: **demand-weighted** importance (21), **peak-hour** importance (20), **passenger-time cost when the station closes** (22), and **multimodal interchange** role (17).

These are not the same question, and they do not have to give the same answer. A rural stop can be a cut vertex while carrying eight trips a day; a downtown platform can carry thousands of trips a day and be trivially replaceable because three identical platforms sit across the street. This notebook is the synthesis step: it loads every lens that is actually available, ranks the stations under each, measures **how much the lenses agree** (Spearman correlation between lens rankings, top-50 overlap counts, and heatmaps of both), and then separates the stations that are critical under **many** lenses - the robustly critical ones - from the stations that are critical under exactly **one** lens, which are usually artifacts of that lens's definition rather than genuine national vulnerabilities.

The deliverable is a ranked shortlist of nationally critical stations, each with an explicit reason for why it qualifies. That is the artefact a transport planner would actually act on: not a centrality column, but a short list of names with a justification attached to each.

**Research question addressed here:** do the different definitions of "critical station" used across this project converge on the same stations, and which stations survive all of them?

## Inputs

The notebook is **degradation-tolerant**: only the first input is mandatory, and every other lens is loaded if its notebook has run and skipped with an explicit message if it has not.

| Lens | Source table | Notebook | Required? |
|---|---|---|---|
| `betweenness` | `04*/tables/stop_metrics.csv` (`approx_betweenness`) | 04 | **yes** (also supplies the station universe) |
| `service_volume` | `04*/tables/stop_metrics.csv` (`weighted_degree`) | 04 | **yes** |
| `articulation` | `03*/tables/articulation_points.csv` + `02*/tables/edges.csv` | 03, 02 | optional |
| `no_walk_alternative` | `05*/tables/critical_isolation.csv` (`nearest_alt_m`) | 05 | optional |
| `demand_weighted` | `21*/tables/demand_weighted_criticality.csv` | 21 | optional |
| `peak_hour` | `20*/tables/peak_vs_offpeak_centrality.csv` (+ `19*/tables/window_summary.csv`) | 20, 19 | optional |
| `closure_time_cost` | `22*/tables/rerouting_results.csv` | 22 | optional |
| `multimodal` | `17*/tables/transfer_hubs.csv` (`n_modes`) | 17 | optional |

**No raw GTFS is read here.** `stop_times.txt` is never touched, so there is no external download and no 816 MB streaming step. Everything is a join over tables that earlier stages already wrote.

## Outputs

Everything is written under `outputs/nb/23_critical_station_lenses/`:

* `tables/critical_station_lenses.csv` - long format, one row per (station, lens): `stop_id, stop_name, lens, rank, score`.
* `tables/lens_agreement.csv` - one row per lens pair: `lens_a, lens_b, spearman_rho, top50_overlap`.
* `tables/lens_agreement_detail.csv` - the same pairs plus `n_common_stations`, so a weak correlation computed on a thin overlap cannot be mistaken for a strong one.
* `tables/lens_coverage.csv` - what each lens measures, where it came from, and how many of the 30k stations it can actually score.
* `tables/station_lens_profile.csv` - every station with its rank and percentile under every lens.
* `tables/critical_station_shortlist.csv` - the final ranked planner-facing shortlist with a `reasons` column.
* `tables/lens_specific_stations.csv` - stations flagged by exactly one lens (candidate artifacts).
* `lens_synthesis_summary.json` - headline numbers.
* `figures/lens_spearman_heatmap.png`, `lens_top_overlap_heatmap.png`, `stations_by_lens_count.png`, `shortlist_lens_profile.png`, `shortlist_map.png`, `lens_specific_counts.png`.

Nothing outside `outputs/nb/23_critical_station_lenses/` is written. The report-cited folders `outputs/tables`, `outputs/figures` and `outputs/rail` are never touched.

## Notebooks that must run first

**Mandatory:** `04_centrality_analysis` (which itself needs `01` and `02`).
**Strongly recommended:** `02_graph_construction` (for exact articulation severity), `03_descriptive_analysis`, `05_critical_station_isolation`.
**Optional, and the reason to re-run this notebook later:** `17`, `19`, `20`, `21`, `22`. With only `02`-`05` present the notebook still runs and produces a four-lens comparison; each extension notebook that has run adds one more column to the matrix.

## 1. Environment bootstrap

The cell below makes the notebook runnable both on a local checkout and on Google Colab. It defines `_ensure(...)`, which pip-installs only the packages that are genuinely missing (so re-running the notebook is cheap), and `find_repo_root()`, which walks up from the current directory looking for the GTFS folder and, failing that, clones the repository into `/content`. It then sets `REPO`, `DATA` and `OUT` and creates the notebook output root. Every later cell relies on these three paths, so this must run first.

In [ ]:
# --- Environment bootstrap (safe to re-run, works locally and on Google Colab) ---
import os, sys, subprocess
from pathlib import Path

def _ensure(*pkgs):
    """Install only the packages that are actually missing."""
    import importlib.util
    alias = {"scikit-learn": "sklearn", "python-louvain": "community",
             "python-bidi": "bidi", "node2vec": "node2vec"}
    missing = [p for p in pkgs
               if importlib.util.find_spec(alias.get(p, p.replace("-", "_"))) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def find_repo_root():
    """Find the repo locally; on Colab, clone it."""
    here = Path(os.getcwd()).resolve()
    for cand in [here, *here.parents]:
        if (cand / "israel-public-transportation").is_dir():
            return cand
    target = Path("/content/israel-transit-network-resilience")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/seanfourman/israel-transit-network-resilience.git",
                        str(target)], check=True)
    return target

REPO = find_repo_root()
os.chdir(REPO)
DATA = REPO / "israel-public-transportation"
OUT = REPO / "outputs" / "nb"
OUT.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO)

## 2. Libraries, stage folders and tunable constants

We import the scientific stack and fix the folder layout: this notebook owns `outputs/nb/23_critical_station_lenses/` with `tables/` and `figures/` sub-folders, and reads every other stage read-only.

All the knobs live here so a grader can change behaviour in one place:

* `TOP_K = 50` defines what "critical under a lens" means - inside that lens's top 50. The output column mandated by the project's data contract is literally called `top50_overlap`, so if you change `TOP_K` the column name stops matching its contents; the notebook prints a warning if you do.
* `MIN_LENSES_FOR_SHORTLIST = 2` stops a station from reaching the shortlist on the strength of a single lens that happens to be the only one covering it.
* `AP_SEVERITY_EXACT` is the only real cost knob. When `True` the notebook computes, for every articulation point, exactly how many stations end up outside the largest surviving fragment after that station is deleted. That is one graph traversal per cut vertex - roughly 900 traversals of a 30k-node / 52k-edge graph, about **30-60 seconds** in pure Python. Setting it to `False` falls back to ranking cut vertices by degree, which is fast but a genuinely worse proxy, and the notebook says so out loud when it does that.
* `UNREACHABLE_PENALTY_SECONDS` only matters if notebook 22 has run; see section 11 for why it exists and why it is an assumption rather than a measurement.

In [ ]:
# --- Libraries and stage folders ------------------------------------------
_ensure('pandas', 'numpy', 'matplotlib', 'seaborn', 'scipy')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

sns.set_theme(style='whitegrid', font_scale=1.05)

STAGE = OUT / '23_critical_station_lenses'   # everything this notebook produces
TABLES = STAGE / 'tables'
FIGURES = STAGE / 'figures'
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

# --- Tunable constants ----------------------------------------------------
TOP_K = 50                          # "critical under a lens" = inside that lens's top K
MIN_LENSES_FOR_SHORTLIST = 2        # a station must be evaluated by >= this many lenses
SHORTLIST_SIZE = 40                 # rows in the final planner-facing shortlist
MIN_COMMON_FOR_RHO = 20             # skip Spearman when two lenses share fewer stations
AP_SEVERITY_EXACT = True            # exact cut-vertex severity (~30-60 s; see section 6)
UNREACHABLE_PENALTY_SECONDS = 3600  # cost charged per unit of unreachable demand (nb 22)
FIG_DPI = 150                       # figure resolution; drop to 90 for faster, smaller files
TOP_N_FIG = 25                      # rows in the profile heatmap and the bar figures

if TOP_K != 50:
    print(f'WARNING: TOP_K={TOP_K} but the contracted column name is "top50_overlap"; '
          'the column will hold top-' + str(TOP_K) + ' overlaps despite its name.')
print('this stage :', STAGE)

## 3. Hebrew label rendering

Stop names in the Israeli GTFS feed are Hebrew, and several figures below print them (the shortlist profile heatmap in particular). Matplotlib does not implement the Unicode bidirectional algorithm, so right-to-left text comes out reversed and unreadable. The cell below monkey-patches `matplotlib.text.Text.set_text` once so that any string containing Hebrew characters is converted to display order via `python-bidi` before it is drawn, and selects a font that actually has Hebrew glyphs (Arial on Windows, DejaVu Sans everywhere else). It is idempotent - re-running it will not stack patches. All other text in the notebook is English, per the submission requirement.

In [ ]:
# Stop names are Hebrew. Matplotlib does not apply the Unicode bidi algorithm, so
# Hebrew labels render reversed. Patch it once, before drawing any figure.
_ensure("python-bidi")
import re
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.text as mtext
from bidi.algorithm import get_display

_HEBREW_RE = re.compile(r"[\u0590-\u05FF]")

def fix_he(text):
    """Return display-ordered text. Non-Hebrew is returned untouched."""
    if not isinstance(text, str) or not _HEBREW_RE.search(text):
        return text
    return get_display(text)

def install_hebrew():
    # Arial exists on Windows; DejaVu Sans ships with matplotlib and covers Hebrew.
    matplotlib.rcParams["font.family"] = ["Arial", "DejaVu Sans"]
    matplotlib.rcParams["axes.unicode_minus"] = False
    if getattr(mtext.Text, "_bidi_patched", False):
        return
    _orig = mtext.Text.set_text
    def set_text(self, s):
        if isinstance(s, str) and getattr(self, "_bidi_display", None) == s:
            return _orig(self, s)
        fixed = fix_he(s)
        if isinstance(fixed, str):
            self._bidi_display = fixed
        return _orig(self, fixed)
    mtext.Text.set_text = set_text
    mtext.Text._bidi_patched = True

install_hebrew()

## 4. Locating upstream stages

This notebook reads up to eight other stages, and half of them may not exist yet. The helpers below resolve a stage folder by its **two-digit prefix** rather than its exact slug (`OUT.glob('04*')`), so a stage folder renamed from `04_centrality_analysis` to `04_centrality` still resolves, and they search recursively inside it so it does not matter whether a stage put its CSVs at the folder root or under `tables/`.

There are two loaders, and the difference between them is deliberate:

* `require_table` raises a `FileNotFoundError` that names the notebook you have to run first. It is used only for notebook 04, which supplies the station universe that every other lens is projected onto.
* `optional_table` prints a one-line `[skipped]` message naming the missing file and the notebook that would produce it, records the gap in `MISSING_LENSES`, and returns `None`. The lens is then simply absent from the comparison - the notebook does not fabricate a substitute.

Every identifier column is forced to `str` on read. Israeli GTFS stop IDs are numeric-looking, so a single table read without `dtype=str` would silently produce integer keys and every merge against it would come back empty.

In [ ]:
# --- Stage resolution and loading helpers ---------------------------------
ID_COLUMNS = ['stop_id', 'from_stop', 'to_stop', 'removed_stop']
MISSING_LENSES = []      # (lens label, notebook that would provide it, what was missing)


def stage_dir(prefix):
    """Resolve a stage folder by two-digit prefix, e.g. '04' -> 04_centrality_analysis."""
    matches = sorted(p for p in OUT.glob(prefix + '*') if p.is_dir())
    return matches[0] if matches else None


def find_artifact(prefix, filename):
    """Full path of `filename` inside the stage folder with this prefix, or None."""
    folder = stage_dir(prefix)
    if folder is None:
        return None
    direct = folder / filename
    if direct.exists():
        return direct
    matches = sorted(folder.rglob(filename))
    return matches[0] if matches else None


def _read(path):
    """Read a stage CSV, forcing every known identifier column to string."""
    return pd.read_csv(path, dtype={c: str for c in ID_COLUMNS}, encoding='utf-8-sig')


def require_table(prefix, filename, notebook):
    """Load a mandatory upstream table, or fail with an actionable message."""
    path = find_artifact(prefix, filename)
    if path is None:
        raise FileNotFoundError(
            f'{filename} was not found under outputs/nb/{prefix}* - run notebook '
            f'{notebook} first; it is the stage that writes {filename}.')
    df = _read(path)
    print(f'[required] {filename:<38} {len(df):>7,} rows  <-  {path}')
    return df


def optional_table(prefix, filename, notebook, lens_label):
    """Load an optional upstream table; return None with a clear message if absent."""
    path = find_artifact(prefix, filename)
    if path is None:
        MISSING_LENSES.append((lens_label, notebook, filename))
        print(f'[skipped ] lens "{lens_label}" unavailable: {filename} not found under '
              f'outputs/nb/{prefix}* - run notebook {notebook} to enable it.')
        return None
    df = _read(path)
    print(f'[loaded  ] {filename:<38} {len(df):>7,} rows  <-  {path}')
    return df


print('stages currently present under', OUT)
for folder in sorted(p.name for p in OUT.iterdir() if p.is_dir()):
    print('   ', folder)

## 5. The station universe (spine)

Every lens has to be projected onto one shared set of stations, otherwise "rank 3 under lens A" and "rank 3 under lens B" are not comparable. We use notebook 04's `stop_metrics.csv` as the spine because it covers exactly the active stops of the trip-adjacency graph (~30,463 stations) and carries the name, coordinates and region we need for the figures and for the final shortlist.

A lens may cover fewer stations than the spine - that is expected and is the single most important caveat in this notebook. The isolation lens from notebook 05, for example, only evaluated a pre-filtered subset of stops. Any station a lens does not cover is recorded as **not evaluated**, never as "scored zero": treating an unevaluated station as unimportant would quietly convert a coverage gap into a finding.

In [ ]:
# --- Station universe from notebook 04 ------------------------------------
metrics = require_table('04', 'stop_metrics.csv', '04_centrality_analysis')
metrics['stop_id'] = metrics['stop_id'].astype(str).str.strip()
metrics = metrics.drop_duplicates(subset='stop_id')

for needed in ('approx_betweenness', 'weighted_degree', 'stop_name', 'lat', 'lon'):
    if needed not in metrics.columns:
        raise KeyError(f'stop_metrics.csv has no "{needed}" column - re-run notebook 04; '
                       f'columns found: {list(metrics.columns)}')

keep = ['stop_id', 'stop_name', 'lat', 'lon', 'region']
keep = [c for c in keep if c in metrics.columns]
spine = metrics[keep].copy()
if 'region' not in spine.columns:
    spine['region'] = 'Unknown'

SPINE_IDS = set(spine['stop_id'])
NAME_BY_ID = dict(zip(spine['stop_id'], spine['stop_name']))
print(f'station universe: {len(spine):,} active stops')
spine.head()

## 6. The lens registry

`register_lens` is the one place where a raw upstream column becomes a comparable ranking. It takes a table, an identifier column and a score column, and does five things:

1. **Orients the score.** Every lens is stored so that a *higher score means more critical*. Lenses whose natural metric points the other way (a rank number, where 1 is worst) are passed `higher_is_worse=False` and the stored score is negated. This is why the `score` column in the exported long table is a criticality score rather than a verbatim copy of the upstream metric - the sign convention has to be uniform for any of the comparisons below to mean anything.
2. **Deduplicates** to one row per station (taking the worst value if a table repeats a station).
3. **Restricts to the spine**, and reports how many rows it had to drop because the station is not in the trip-adjacency graph.
4. **Ranks** with `method='min'`, so tied stations share a rank - important because several lenses have long ties (mode counts, small integer detours).
5. **Percentiles** the score within the lens's own covered set, giving a value in (0, 1] that is comparable across lenses of wildly different units (seconds, metres, trips, dimensionless betweenness).

A lens that is missing, empty, or lacking the requested column is skipped with a message and simply does not appear in the comparison.

In [ ]:
# --- Lens registry --------------------------------------------------------
LENSES = {}      # lens name -> DataFrame[stop_id, score, rank, pct]
LENS_META = {}   # lens name -> provenance and coverage record


def register_lens(name, df, id_col, score_col, higher_is_worse=True,
                  question='', source=''):
    """Turn one upstream column into a ranked, percentiled lens over the spine."""
    if df is None:
        return None
    for col in (id_col, score_col):
        if col not in df.columns:
            print(f'[skipped ] lens "{name}": source table has no "{col}" column '
                  f'(found: {list(df.columns)[:8]} ...).')
            MISSING_LENSES.append((name, source, 'column ' + col))
            return None

    sub = df[[id_col, score_col]].copy()
    sub.columns = ['stop_id', 'score']
    sub['stop_id'] = sub['stop_id'].astype(str).str.strip()
    sub['score'] = pd.to_numeric(sub['score'], errors='coerce')
    if not higher_is_worse:
        sub['score'] = -sub['score']          # store so that higher always = more critical
    sub = sub.dropna(subset=['stop_id', 'score'])
    sub = sub.groupby('stop_id', as_index=False)['score'].max()

    before = len(sub)
    sub = sub[sub['stop_id'].isin(SPINE_IDS)]
    dropped = before - len(sub)
    if len(sub) < 2:
        print(f'[skipped ] lens "{name}": fewer than two usable stations after cleaning.')
        MISSING_LENSES.append((name, source, 'no usable rows'))
        return None

    sub['rank'] = sub['score'].rank(ascending=False, method='min').astype(int)
    sub['pct'] = sub['score'].rank(ascending=True, pct=True)
    sub = sub.sort_values('rank').reset_index(drop=True)

    LENSES[name] = sub
    LENS_META[name] = {
        'lens': name,
        'question': question,
        'source': source,
        'metric': score_col,
        'higher_metric_is_more_critical': bool(higher_is_worse),
        'stations_evaluated': int(len(sub)),
        'coverage_share_of_universe': round(len(sub) / len(SPINE_IDS), 4),
        'rows_dropped_outside_universe': int(dropped),
    }
    print(f'[lens ok ] {name:<20} {len(sub):>7,} stations '
          f'({len(sub) / len(SPINE_IDS):.1%} of the universe), metric = {score_col}')
    return sub

## 7. Lens (a) topological betweenness and lens (c) service volume

Both come straight from notebook 04 and both cover the full universe, so they anchor the comparison.

* **`betweenness`** uses `approx_betweenness` - the share of shortest paths passing through a station. It is the classic "if this fails, traffic has to route around it" measure. Note that it is *approximate*: notebook 04 computed it by pivot sampling, and its own stability table shows the ranking is stable at the top but noisy in the tail. Treat rank differences of a few places as meaningless.
* **`service_volume`** uses `weighted_degree` - the total number of scheduled trips touching the station. This is the operational rather than topological reading of importance, and it is the only lens here that reflects how much service actually exists rather than where the station sits in the graph.

These two are deliberately kept separate rather than merged into one "centrality" lens, because the whole point of the notebook is to test whether they agree.

In [ ]:
# --- Lens (a): topological betweenness ------------------------------------
register_lens(
    'betweenness', metrics, 'stop_id', 'approx_betweenness', higher_is_worse=True,
    question='Which stations carry the most shortest paths through the network?',
    source='04_centrality_analysis/tables/stop_metrics.csv')

# --- Lens (c): service volume ---------------------------------------------
register_lens(
    'service_volume', metrics, 'stop_id', 'weighted_degree', higher_is_worse=True,
    question='Which stations carry the largest scheduled trip volume?',
    source='04_centrality_analysis/tables/stop_metrics.csv')

## 8. Lens (b) articulation points, ranked by how much damage they actually do

Notebook 03 lists the ~900 cut vertices, but being a cut vertex is a **binary** property and a binary property cannot be ranked. Most of those 900 are the neck of a short dead-end branch: removing them strands three stops. A handful sit between genuinely large parts of the network. Comparing a binary flag against a continuous lens would also make the Spearman correlations meaningless (one variable would be entirely ties).

So we compute a severity for each cut vertex: **how many stations end up outside the largest surviving fragment when that station is deleted**. The code builds a plain adjacency dictionary from `02*/edges.csv` (undirected projection, self-loops dropped) and, for each cut vertex, runs one iterative depth-first traversal that starts from each of its neighbours with the vertex itself marked as blocked. The visited set is shared across the branches of a single vertex, so each fragment is explored exactly once and the whole traversal costs `O(V + E)` *for that vertex's component only*.

**Cost:** roughly 900 traversals of a 30k-node / 52k-edge graph in pure Python, about **30-60 seconds**. Set `AP_SEVERITY_EXACT = False` to skip it - the notebook then ranks cut vertices by degree instead and prints an explicit warning, because degree is a poor stand-in for severance size (a degree-2 stop can be the only link to an entire town).

This lens's universe is the ~900 cut vertices, not all 30k stations. Correlations involving it therefore answer "among cut vertices, does severance size track the other lenses?" - which is the right question, but a narrower one than it may look.

In [ ]:
# --- Lens (b): articulation points with an exact severance score ------------
ap_df = optional_table('03', 'articulation_points.csv', '03_descriptive_analysis',
                       'articulation')
edges_path = find_artifact('02', 'edges.csv')
ap_score_col = None

if ap_df is not None:
    ap_df = ap_df.copy()
    ap_df['stop_id'] = ap_df['stop_id'].astype(str).str.strip()

if ap_df is not None and edges_path is not None and AP_SEVERITY_EXACT:
    edges_df = _read(edges_path)
    adj = {}
    for u, v in zip(edges_df['from_stop'].astype(str), edges_df['to_stop'].astype(str)):
        if u == v:
            continue                     # self-loops carry no connectivity information
        adj.setdefault(u, set()).add(v)
        adj.setdefault(v, set()).add(u)
    print(f'undirected adjacency built: {len(adj):,} nodes')

    def _explore(start, visited):
        """Iterative DFS from `start`, skipping anything already in `visited`."""
        stack, members = [start], []
        visited.add(start)
        while stack:
            x = stack.pop()
            members.append(x)
            for y in adj[x]:
                if y not in visited:
                    visited.add(y)
                    stack.append(y)
        return members

    comp_size, _seen = {}, set()
    for node in adj:
        if node in _seen:
            continue
        members = _explore(node, _seen)
        for m in members:
            comp_size[m] = len(members)

    def stranded_nodes(v):
        """Stations left outside the largest surviving fragment when v is deleted."""
        blocked, sizes = {v}, []
        for start in adj[v]:
            if start in blocked:
                continue
            stack, count = [start], 0
            blocked.add(start)
            while stack:
                x = stack.pop()
                count += 1
                for y in adj[x]:
                    if y not in blocked:
                        blocked.add(y)
                        stack.append(y)
            sizes.append(count)
        return 0 if not sizes else sum(sizes) - max(sizes)

    ap_df = ap_df[ap_df['stop_id'].isin(adj)].copy()
    ap_df['stranded_nodes'] = [stranded_nodes(s) for s in ap_df['stop_id']]
    ap_df['component_size'] = [comp_size[s] for s in ap_df['stop_id']]
    ap_score_col = 'stranded_nodes'
    ap_df.sort_values('stranded_nodes', ascending=False).head(10).to_csv(
        TABLES / 'top_articulation_severity.csv', index=False, encoding='utf-8-sig')
    print(f'exact severance computed for {len(ap_df):,} cut vertices; '
          f'worst strands {int(ap_df["stranded_nodes"].max()):,} stations, '
          f'median strands {ap_df["stranded_nodes"].median():.0f}')

elif ap_df is not None:
    ap_score_col = 'degree' if 'degree' in ap_df.columns else None
    print('[fallback] ranking cut vertices by DEGREE, not by severance size, because '
          'edges.csv from notebook 02 is missing or AP_SEVERITY_EXACT is False. '
          'Degree is a weak proxy: a degree-2 stop can be the sole link to a whole town.')

if ap_df is not None and ap_score_col is not None:
    register_lens(
        'articulation', ap_df, 'stop_id', ap_score_col, higher_is_worse=True,
        question='Among cut vertices, which one strands the most stations when removed?',
        source='03_descriptive_analysis/tables/articulation_points.csv (+ 02 edges.csv)')

## 9. Lens (d) no walking alternative

Notebook 05 measured, for each station it examined, the straight-line distance to the nearest *other* stop that could absorb its passengers (`nearest_alt_m`). A large value means a passenger stranded there has nowhere to walk to: the station is irreplaceable on the ground, regardless of what the graph says.

**Be honest about what this lens covers.** Notebook 05 did not evaluate all ~30k stations; it evaluated a pre-filtered set of stops it had already flagged as critical (about 3,000 rows), and within those, the overwhelming majority have an alternative within 300 m. So this lens is *conditional*: it ranks irreplaceability **among stations that were already considered important**, and it says nothing at all about the other ~27k stations. Two consequences follow, and both are handled explicitly later: its correlations with other lenses are computed on that subset only, and a station absent from it is recorded as unevaluated rather than as "has an alternative".

In [ ]:
# --- Lens (d): distance to the nearest substitutable stop -------------------
iso = optional_table('05', 'critical_isolation.csv', '05_critical_station_isolation',
                     'no_walk_alternative')
if iso is not None:
    print(f'coverage note: notebook 05 evaluated {len(iso):,} pre-filtered stops, i.e. '
          f'{len(iso) / len(SPINE_IDS):.1%} of the universe - this lens is conditional '
          'on a station already having been flagged critical upstream.')
    if 'is_isolated' in iso.columns:
        flagged = int(pd.Series(iso['is_isolated']).astype(str).str.lower()
                      .isin(['true', '1', 'yes']).sum())
        print(f'   of those, {flagged:,} were flagged as genuinely isolated upstream.')

register_lens(
    'no_walk_alternative', iso, 'stop_id', 'nearest_alt_m', higher_is_worse=True,
    question='Which stations have no substitutable stop within walking distance?',
    source='05_critical_station_isolation/tables/critical_isolation.csv')

## 10. Lens (e) demand-weighted criticality

Notebook 21 re-ranks stations by combining topological importance with the population the station serves, so that a structurally interesting stop in an empty area drops and a mundane stop serving 40,000 people rises. We prefer its `demand_weighted_rank` column (a rank, where 1 is most critical - hence `higher_is_worse=False`, which flips the sign so the stored score keeps the notebook-wide convention). If only the intermediate `demand_proxy.csv` exists we fall back to its `demand_weight` column.

If notebook 21 has not run, this lens is skipped. We deliberately do **not** improvise a substitute out of the socioeconomic table from notebook 08: population attached to a stop by a spatial join is exactly the proxy notebook 21 is responsible for building and validating, and rebuilding it here with different assumptions would produce a second, silently inconsistent version of the same quantity.

In [ ]:
# --- Lens (e): demand-weighted criticality ---------------------------------
dem = optional_table('21', 'demand_weighted_criticality.csv',
                     '21_demand_weighted_criticality', 'demand_weighted')

if dem is not None and 'demand_weighted_rank' in dem.columns:
    register_lens(
        'demand_weighted', dem, 'stop_id', 'demand_weighted_rank', higher_is_worse=False,
        question='Which stations matter most once the population served is weighted in?',
        source='21_demand_weighted_criticality/tables/demand_weighted_criticality.csv')
else:
    proxy = optional_table('21', 'demand_proxy.csv', '21_demand_weighted_criticality',
                           'demand_weighted (proxy fallback)')
    if proxy is not None:
        print('[fallback] using demand_proxy.demand_weight; this is raw served demand, '
              'not the demand-weighted criticality ranking itself.')
    register_lens(
        'demand_weighted', proxy, 'stop_id', 'demand_weight', higher_is_worse=True,
        question='Which stations serve the most demand (proxy fallback)?',
        source='21_demand_weighted_criticality/tables/demand_proxy.csv')

## 11. Lens (f) peak-hour criticality

Notebook 20 recomputes centrality separately for each time window built in notebook 19, because the network at 08:00 is not the network at 23:00 - branches that only run at rush hour exist in one and not the other. This lens takes the **peak** window only.

Picking that window has to be robust to whatever naming notebook 19 chose, so the selection is: prefer window names containing `peak` but not `off`; if several qualify, use notebook 19's `window_summary.csv` to take the one with the most trips; if that table is unavailable, take the first candidate alphabetically. The chosen window is printed, so the choice is never silent. Within the window we rank by `approx_betweenness`, falling back to `weighted_degree` if the betweenness column is not present.

In [ ]:
# --- Lens (f): peak-hour criticality ---------------------------------------
peak = optional_table('20', 'peak_vs_offpeak_centrality.csv', '20_dynamic_resilience',
                      'peak_hour')
peak_window = None

if peak is not None and 'window' in peak.columns:
    windows = sorted({str(w) for w in peak['window'].dropna()})
    named = [w for w in windows if 'peak' in w.lower() and 'off' not in w.lower()]
    candidates = named or windows
    ws_path = find_artifact('19', 'window_summary.csv')
    if len(candidates) > 1 and ws_path is not None:
        ws = _read(ws_path)
        ws['window'] = ws['window'].astype(str)
        busiest = ws[ws['window'].isin(candidates)]
        if 'trips' in busiest.columns and len(busiest):
            candidates = [busiest.sort_values('trips', ascending=False).iloc[0]['window']]
    peak_window = str(candidates[0])
    print(f'windows available: {windows}')
    print(f'peak window selected: "{peak_window}"')
    peak = peak[peak['window'].astype(str) == peak_window].copy()

peak_metric = 'weighted_degree'
if peak is not None and 'approx_betweenness' in peak.columns:
    peak_metric = 'approx_betweenness'

register_lens(
    'peak_hour', peak, 'stop_id', peak_metric, higher_is_worse=True,
    question='Which stations are critical specifically during the peak window?',
    source='20_dynamic_resilience/tables/peak_vs_offpeak_centrality.csv'
           + (f' [window={peak_window}]' if peak_window else ''))

## 12. Lens (g) passenger-time cost of closing the station

Notebook 22 closes one station at a time on the travel-time graph and measures how much longer everyone else's journey becomes. This is the closest thing in the project to a real impact metric, because it is denominated in seconds of passenger time rather than in graph units.

There is one modelling decision that has to be stated openly. `mean_detour_seconds` is an average over the pairs that are **still reachable** after the closure. A station whose removal disconnects a whole region can therefore post a *low* mean detour, simply because the worst-affected journeys dropped out of the average entirely - the metric rewards catastrophic failure. To stop that inversion we charge unreachability explicitly:

```
closure_cost_seconds = mean_detour_seconds + UNREACHABLE_PENALTY_SECONDS * (1 - reachable_share)
```

`UNREACHABLE_PENALTY_SECONDS = 3600` says "treat a journey that becomes impossible as costing an hour". That number is an **assumption, not a measurement** - the true cost of an impossible journey is unbounded. It is a named constant precisely so that its influence can be tested: raise it and disconnecting stations dominate the lens, set it to 0 and you get the raw (inverted) mean detour. If notebook 22 does not export `reachable_share`, the raw mean detour is used and the caveat above stands unmitigated.

In [ ]:
# --- Lens (g): passenger-time cost under closure ---------------------------
rr = optional_table('22', 'rerouting_results.csv', '22_rerouting_analysis',
                    'closure_time_cost')

if rr is not None and 'mean_detour_seconds' in rr.columns:
    rr = rr.copy()
    cost = pd.to_numeric(rr['mean_detour_seconds'], errors='coerce').fillna(0.0)
    if 'reachable_share' in rr.columns:
        lost = 1.0 - pd.to_numeric(rr['reachable_share'], errors='coerce').clip(0, 1).fillna(1.0)
        cost = cost + UNREACHABLE_PENALTY_SECONDS * lost
        print(f'unreachable demand charged at {UNREACHABLE_PENALTY_SECONDS:,} s '
              '(an assumption, see the markdown above)')
    else:
        print('[caveat  ] rerouting_results.csv has no reachable_share column, so the '
              'mean detour is used raw - stations that disconnect the network are '
              'under-rated by this lens.')
    rr['closure_cost_seconds'] = cost

register_lens(
    'closure_time_cost', rr, 'removed_stop', 'closure_cost_seconds', higher_is_worse=True,
    question='Which closures cost passengers the most travel time?',
    source='22_rerouting_analysis/tables/rerouting_results.csv')

## 13. Lens (h) multimodal interchange

Notebook 17 counts how many distinct modes (bus, rail, light rail, trolleybus, cable tram, demand-responsive) meet at each stop. A station where four modes interchange is a different kind of critical from a station with high betweenness: losing it does not merely lengthen paths, it breaks the transfer that makes a multi-leg journey possible at all, and there is usually no second place nearby where the same two modes meet.

`n_modes` is a small integer, so this lens has very heavy ties - most stops are 1, a few hundred are 2, a handful are 3 or more. Rank ties are handled with `method='min'`, but the practical effect is that its "top 50" is an arbitrary slice of a large tied group. Read its overlap counts with that in mind; the correlation coefficient is more informative than the top-50 count for this lens.

In [ ]:
# --- Lens (h): multimodal interchange --------------------------------------
hubs = optional_table('17', 'transfer_hubs.csv', '17_multimodal_transfer_hubs',
                      'multimodal')

if hubs is not None and 'n_modes' in hubs.columns:
    counts = pd.to_numeric(hubs['n_modes'], errors='coerce').value_counts().sort_index()
    print('modes served -> number of stops:')
    print(counts.to_string())

register_lens(
    'multimodal', hubs, 'stop_id', 'n_modes', higher_is_worse=True,
    question='Which stations are the interchange point between several modes?',
    source='17_multimodal_transfer_hubs/tables/transfer_hubs.csv')

## 14. What we ended up with

Before any comparison, we write down exactly which lenses exist in this run and how much of the network each can see. `lens_coverage.csv` is the honesty record of the notebook: every correlation and every shortlist rank below has to be read against it, because a lens covering 3,000 stations and a lens covering 30,000 are not equally informative even when they produce the same-looking number.

The comparison needs at least two lenses to mean anything, so the cell stops with a clear message if fewer than two survived.

In [ ]:
# --- Lens inventory --------------------------------------------------------
if len(LENSES) < 2:
    raise RuntimeError(
        f'only {len(LENSES)} lens/lenses could be built, so there is nothing to compare. '
        'Run notebooks 03, 04 and 05 first (04 is mandatory), then re-run this notebook.')

lens_names = list(LENSES)
coverage = (pd.DataFrame([LENS_META[n] for n in lens_names])
            .sort_values('stations_evaluated', ascending=False)
            .reset_index(drop=True))
coverage.to_csv(TABLES / 'lens_coverage.csv', index=False, encoding='utf-8-sig')

print(f'{len(lens_names)} lenses available: {lens_names}')
if MISSING_LENSES:
    print('\nlenses NOT available in this run:')
    for label, notebook, what in MISSING_LENSES:
        print(f'   {label:<32} needs {notebook}  (missing: {what})')
else:
    print('all eight lenses were available.')

coverage[['lens', 'metric', 'stations_evaluated', 'coverage_share_of_universe']]

## 15. The long table: every station under every lens

`critical_station_lenses.csv` is the contracted output of this stage and the raw material for everything after it: one row per (station, lens) pair with the rank and the criticality score, in long format so that adding a ninth lens later changes the row count and not the schema. Remember that `score` follows the notebook-wide convention - higher is always more critical - so for a lens whose upstream metric was a rank number the stored score is its negation. The `rank` column is the human-readable one: 1 is the most critical station under that lens.

In [ ]:
# --- Long table: one row per (station, lens) -------------------------------
long_parts = []
for name in lens_names:
    part = LENSES[name][['stop_id', 'rank', 'score']].copy()
    part.insert(1, 'lens', name)
    long_parts.append(part)

lenses_long = pd.concat(long_parts, ignore_index=True)
lenses_long['stop_name'] = lenses_long['stop_id'].map(NAME_BY_ID)
lenses_long = (lenses_long[['stop_id', 'stop_name', 'lens', 'rank', 'score']]
               .sort_values(['lens', 'rank'])
               .reset_index(drop=True))
lenses_long.to_csv(TABLES / 'critical_station_lenses.csv', index=False, encoding='utf-8-sig')

print(f'critical_station_lenses.csv: {len(lenses_long):,} rows '
      f'({lenses_long["stop_id"].nunique():,} distinct stations x {len(lens_names)} lenses)')
lenses_long.groupby('lens').head(3).head(24)

## 16. Do the lenses agree? Spearman correlation and top-K overlap

For each pair of lenses we compute two very different things, on purpose:

* **Spearman rank correlation** over the stations *both* lenses evaluated. It answers "do these two orderings agree across the whole network?". It is computed pairwise-complete, so a pair involving a narrow lens is computed on that narrow overlap only - which is why `n_common_stations` is exported alongside it in `lens_agreement_detail.csv`, and why pairs with fewer than `MIN_COMMON_FOR_RHO` stations in common are left blank rather than reported as a coincidence.
* **Top-K overlap**: how many of the same stations appear in both lenses' top 50. This answers the question a planner actually asks, which is not "are the orderings similar in general" but "do the two definitions point me at the same handful of stations". The two can diverge sharply: two lenses can correlate at rho = 0.8 across 30,000 stations and still share only a handful of names at the very top, because rank correlation is dominated by the enormous, uninteresting middle of the distribution.

Spearman is used rather than Pearson because the lens scores are in incompatible units (seconds, metres, trips, mode counts) and several are heavily skewed; only the ordering is comparable.

In [ ]:
# --- Pairwise agreement between lenses -------------------------------------
topk_sets = {n: set(LENSES[n].sort_values('rank')['stop_id'].head(TOP_K))
             for n in lens_names}

pair_rows = []
for i, a in enumerate(lens_names):
    for b in lens_names[i + 1:]:
        da = LENSES[a][['stop_id', 'score']].rename(columns={'score': 'score_a'})
        db = LENSES[b][['stop_id', 'score']].rename(columns={'score': 'score_b'})
        both = da.merge(db, on='stop_id', how='inner')
        if len(both) >= MIN_COMMON_FOR_RHO:
            rho = float(spearmanr(both['score_a'], both['score_b'])[0])
        else:
            rho = float('nan')
        pair_rows.append({
            'lens_a': a,
            'lens_b': b,
            'spearman_rho': round(rho, 4),
            'top50_overlap': int(len(topk_sets[a] & topk_sets[b])),
            'n_common_stations': int(len(both)),
        })

agreement = pd.DataFrame(pair_rows)
# contracted file: exactly the four agreed columns
agreement[['lens_a', 'lens_b', 'spearman_rho', 'top50_overlap']].to_csv(
    TABLES / 'lens_agreement.csv', index=False, encoding='utf-8-sig')
# diagnostic file: the same pairs plus the overlap size the rho was computed on
agreement.to_csv(TABLES / 'lens_agreement_detail.csv', index=False, encoding='utf-8-sig')

thin = agreement[agreement['spearman_rho'].isna()]
if len(thin):
    print(f'{len(thin)} lens pair(s) share fewer than {MIN_COMMON_FOR_RHO} stations; '
          'their correlation is left blank rather than reported.')
agreement.sort_values('top50_overlap', ascending=False)

## 17. Heatmaps of agreement

Two square matrices built from the pair table above. The left one is the Spearman matrix on a diverging red-blue scale fixed to [-1, 1], so the colour is comparable between runs and a blank cell means "too little overlap to say" rather than "zero correlation". The right one counts shared members of the top 50; its diagonal is the size of each lens's own top-K set, which is normally 50 but can be smaller for a lens that covers fewer than 50 stations.

Reading them together is the point of the section. High correlation with low top-50 overlap means the lenses broadly agree about the country but disagree about the extremes - and it is the extremes that get maintenance budgets.

In [ ]:
# --- Agreement heatmaps ----------------------------------------------------
rho_m = pd.DataFrame(np.nan, index=lens_names, columns=lens_names, dtype=float)
ovl_m = pd.DataFrame(0, index=lens_names, columns=lens_names, dtype=int)
for _, r in agreement.iterrows():
    a, b = r['lens_a'], r['lens_b']
    rho_m.loc[a, b] = rho_m.loc[b, a] = r['spearman_rho']
    ovl_m.loc[a, b] = ovl_m.loc[b, a] = int(r['top50_overlap'])
for n in lens_names:
    rho_m.loc[n, n] = 1.0
    ovl_m.loc[n, n] = len(topk_sets[n])

fig, ax = plt.subplots(figsize=(1.6 * len(lens_names) + 3, 1.3 * len(lens_names) + 2))
sns.heatmap(rho_m, annot=True, fmt='.2f', cmap='RdBu_r', vmin=-1, vmax=1, center=0,
            linewidths=0.5, cbar_kws={'label': 'Spearman rho'}, ax=ax)
ax.set_title('Agreement between criticality lenses\n'
             '(rank correlation on the stations both lenses evaluated)')
plt.tight_layout()
plt.savefig(FIGURES / 'lens_spearman_heatmap.png', dpi=FIG_DPI)
plt.show()

fig, ax = plt.subplots(figsize=(1.6 * len(lens_names) + 3, 1.3 * len(lens_names) + 2))
sns.heatmap(ovl_m, annot=True, fmt='d', cmap='YlOrRd', linewidths=0.5,
            cbar_kws={'label': f'stations shared in the top {TOP_K}'}, ax=ax)
ax.set_title(f'Top-{TOP_K} overlap between criticality lenses\n'
             '(diagonal = size of each lens own top set)')
plt.tight_layout()
plt.savefig(FIGURES / 'lens_top_overlap_heatmap.png', dpi=FIG_DPI)
plt.show()

## 18. Per-station profile: robustly critical vs lens-specific

Now we pivot the long table into one row per station and attach three summary numbers:

* **`n_lenses_evaluated`** - how many lenses could score this station at all. This is the coverage denominator, and without it the next two numbers are unreadable.
* **`n_lenses_topk`** - in how many lenses' top 50 the station appears. This is the *evidence count* and the primary sort key. Here a station that a lens never evaluated correctly counts as "not flagged by that lens", because the planner-facing claim is "how many independent definitions of critical pick this station out".
* **`consensus_mean_pct`** - the mean percentile across the lenses that did evaluate it, used only as a tie-breaker. This one is *coverage-conditional* and must not be read as an overall score: a station seen by a single narrow lens can score 0.99 here on the strength of one opinion, which is exactly why the shortlist below also requires `n_lenses_evaluated >= MIN_LENSES_FOR_SHORTLIST`.

The two failure modes we are separating are: **robustly critical** (several unrelated definitions independently point at the same station) and **lens-specific** (exactly one lens flags it while others that did look at it did not - usually an artifact of that lens's definition, occasionally a genuine blind spot in all the others).

In [ ]:
# --- Wide per-station profile ----------------------------------------------
rank_wide = pd.concat([LENSES[n].set_index('stop_id')['rank'].rename(n)
                       for n in lens_names], axis=1)
pct_wide = pd.concat([LENSES[n].set_index('stop_id')['pct'].rename(n)
                      for n in lens_names], axis=1)
rank_wide.index.name = 'stop_id'
pct_wide.index.name = 'stop_id'
flags = rank_wide.le(TOP_K)          # NaN ranks compare False, which is what we want

prof = pd.DataFrame(index=pct_wide.index)
prof['n_lenses_evaluated'] = pct_wide.notna().sum(axis=1).astype(int)
prof['n_lenses_topk'] = flags.sum(axis=1).astype(int)
prof['consensus_mean_pct'] = pct_wide.mean(axis=1).round(4)
prof['lenses_topk'] = flags.apply(
    lambda row: ';'.join([c for c in flags.columns if bool(row[c])]), axis=1)
prof = prof.reset_index()

prof = prof.merge(rank_wide.add_prefix('rank_').reset_index(), on='stop_id', how='left')
prof = prof.merge(pct_wide.add_prefix('pct_').reset_index(), on='stop_id', how='left')
prof = prof.merge(spine, on='stop_id', how='left')


def reason_text(row):
    """Human-readable justification: which lenses flagged this station, and at what rank."""
    bits = [f'{n} (#{int(row["rank_" + n])})'
            for n in lens_names
            if pd.notna(row['rank_' + n]) and row['rank_' + n] <= TOP_K]
    return '; '.join(bits) if bits else 'in no lens top-' + str(TOP_K)


prof['reasons'] = prof.apply(reason_text, axis=1)
prof = prof.sort_values(['n_lenses_topk', 'consensus_mean_pct'],
                        ascending=[False, False]).reset_index(drop=True)
prof.to_csv(TABLES / 'station_lens_profile.csv', index=False, encoding='utf-8-sig')

dist = prof['n_lenses_topk'].value_counts().sort_index()
print('stations flagged by exactly k lenses:')
for k, v in dist.items():
    print(f'   k = {k}: {v:,} stations')
print(f'\nflagged by at least one lens: {int((prof["n_lenses_topk"] > 0).sum()):,}')
print(f'flagged by two or more lenses: {int((prof["n_lenses_topk"] >= 2).sum()):,}')

## 19. How concentrated is the evidence?

A bar chart of how many stations are flagged by exactly one lens, exactly two, and so on. The shape of this chart is the headline result of the notebook. If almost every flagged station is flagged by exactly one lens, then "critical" is essentially a property of the measurement rather than of the station, and no single ranking in this project should be presented as *the* answer. If a visible group is flagged by three or more, those stations are the robust core - and the maximum possible value is the number of lenses available in this run, which the chart notes in its title.

In [ ]:
# --- Distribution of evidence across lenses --------------------------------
flagged = prof[prof['n_lenses_topk'] > 0]
counts = flagged['n_lenses_topk'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar([str(k) for k in counts.index], counts.to_numpy(), color='#2563eb')
for bar, val in zip(bars, counts.to_numpy()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f'{val:,}',
            ha='center', va='bottom', fontsize=10)
ax.set_xlabel(f'Number of lenses placing the station in their top {TOP_K}')
ax.set_ylabel('Number of stations')
ax.set_title(f'How many stations are critical under how many lenses\n'
             f'({len(lens_names)} lenses available in this run)')
ax.margins(y=0.14)
plt.tight_layout()
plt.savefig(FIGURES / 'stations_by_lens_count.png', dpi=FIG_DPI)
plt.show()

## 20. The shortlist

The practical deliverable. A station reaches the shortlist only if it was evaluated by at least `MIN_LENSES_FOR_SHORTLIST` lenses (so nothing gets in on a single narrow opinion) and appears in at least one lens's top 50. The ordering is: number of lenses flagging it, then mean percentile as a tie-break.

Every row carries a `reasons` string listing exactly which lenses flagged it and at what rank, so the list is auditable - a planner can see that a station is here because it is a cut vertex stranding 400 stops *and* the busiest interchange in its region, versus a station that is here purely on betweenness. The per-lens `rank_*` columns are kept in the exported file for the same reason.

This is a shortlist, not a budget. It ranks stations by how much independent evidence exists that they matter - not by repair cost, not by failure probability, and not by anything the GTFS feed cannot see (station layout, staffing, actual ridership counts).

In [ ]:
# --- Final ranked shortlist ------------------------------------------------
eligible = prof[(prof['n_lenses_evaluated'] >= MIN_LENSES_FOR_SHORTLIST)
                & (prof['n_lenses_topk'] >= 1)].copy()
shortlist = (eligible.sort_values(['n_lenses_topk', 'consensus_mean_pct'],
                                  ascending=[False, False])
             .head(SHORTLIST_SIZE)
             .reset_index(drop=True))
shortlist.insert(0, 'shortlist_rank', np.arange(1, len(shortlist) + 1))

front = ['shortlist_rank', 'stop_id', 'stop_name', 'region', 'lat', 'lon',
         'n_lenses_topk', 'n_lenses_evaluated', 'consensus_mean_pct', 'reasons']
front = [c for c in front if c in shortlist.columns]
rest = [c for c in shortlist.columns if c not in front]
shortlist = shortlist[front + rest]
shortlist.to_csv(TABLES / 'critical_station_shortlist.csv', index=False, encoding='utf-8-sig')

print(f'{len(eligible):,} stations were eligible; the top {len(shortlist)} are exported.')
shortlist[['shortlist_rank', 'stop_name', 'region', 'n_lenses_topk',
           'consensus_mean_pct', 'reasons']].head(TOP_N_FIG)

## 21. Profile heatmap of the shortlist

One row per shortlisted station, one column per lens, coloured by the station's percentile within that lens (1.0 = the most critical station that lens has ever seen, 0 = the least). Grey gaps are *not evaluated* - the station is outside that lens's coverage - and they are drawn distinctly from a genuine low score, because the two mean opposite things.

A row that is dark all the way across is a station every definition agrees about. A row with one dark cell and the rest pale is a station riding on a single lens, and it is the visual counterpart of the lens-specific table in section 23. Station names are Hebrew and go through the bidi patch installed in section 3; the numeric stop ID is appended because several stops share a name.

In [ ]:
# --- Shortlist profile heatmap ---------------------------------------------
top_rows = shortlist.head(TOP_N_FIG).copy()
pct_cols = ['pct_' + n for n in lens_names]
mat = top_rows[pct_cols].astype(float)
mat.columns = lens_names
mat.index = [f'{name} ({sid})' for name, sid
             in zip(top_rows['stop_name'].fillna('?'), top_rows['stop_id'])]

fig, ax = plt.subplots(figsize=(1.5 * len(lens_names) + 6, 0.42 * len(mat) + 3))
sns.heatmap(mat, annot=True, fmt='.2f', cmap='magma_r', vmin=0, vmax=1,
            linewidths=0.4, linecolor='white',
            cbar_kws={'label': 'percentile within the lens (1.0 = most critical)'},
            mask=mat.isna(), ax=ax)
ax.set_facecolor('#d9d9d9')          # grey = the lens never evaluated this station
ax.set_title(f'Top {len(mat)} shortlisted stations under every available lens\n'
             'grey = not evaluated by that lens (not "low score")')
ax.set_xlabel('lens')
plt.setp(ax.get_yticklabels(), rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig(FIGURES / 'shortlist_lens_profile.png', dpi=FIG_DPI)
plt.show()

## 22. Where the shortlist is

The shortlist plotted over the full station cloud, sized and coloured by how many lenses flagged each station. This is a plain lat/lon scatter rather than a projected map, so the aspect ratio is set to `1 / cos(mean latitude)` to keep the country's shape approximately right instead of horizontally stretched.

The geography matters for interpretation: if the shortlist clusters entirely in the Tel Aviv - Haifa corridor, the synthesis is mostly rediscovering where the network is densest, and the periphery - where a single failure has far fewer alternatives - is being under-served by every lens at once.

In [ ]:
# --- Map of the shortlist --------------------------------------------------
base = spine.dropna(subset=['lat', 'lon'])
pts = shortlist.dropna(subset=['lat', 'lon'])

fig, ax = plt.subplots(figsize=(8, 11))
ax.scatter(base['lon'], base['lat'], s=1.5, color='#cbd5e1', alpha=0.55, linewidths=0)
if len(pts):
    sc = ax.scatter(pts['lon'], pts['lat'], c=pts['n_lenses_topk'],
                    s=40 + 55 * pts['n_lenses_topk'], cmap='plasma_r',
                    edgecolor='black', linewidths=0.5, zorder=3)
    cbar = plt.colorbar(sc, ax=ax, fraction=0.035)
    cbar.set_label(f'lenses placing the station in their top {TOP_K}')
    ax.set_aspect(1 / np.cos(np.deg2rad(float(base['lat'].mean()))))
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title(f'Nationally critical stations shortlist (n = {len(pts)})\n'
             'grey = all other active stops')
plt.tight_layout()
plt.savefig(FIGURES / 'shortlist_map.png', dpi=FIG_DPI)
plt.show()

if 'region' in shortlist.columns:
    print('shortlist by region:')
    print(shortlist['region'].fillna('Unknown').value_counts().to_string())

## 23. Lens-specific stations: the artifacts

The mirror image of the shortlist. These are stations that exactly one lens puts in its top 50 **while at least one other lens looked at them and did not**. That second condition is what makes the finding meaningful: a station flagged by one lens and simply invisible to all the others is a coverage gap, not a disagreement.

Most of these are definitional artifacts, and they are worth naming because each one is a warning about the lens that produced it - a cut vertex on a dead-end branch that no passenger volume justifies, a stop with a huge `nearest_alt_m` because it sits alone in the desert, a multimodal stop that is only multimodal because two route types happen to share a kerb. The counts per lens tell you which lens is the most idiosyncratic: the lens contributing the most single-lens flags is the one whose ranking should least be used on its own.

In [ ]:
# --- Stations flagged by exactly one lens ----------------------------------
single = prof[(prof['n_lenses_topk'] == 1)
              & (prof['n_lenses_evaluated'] >= 2)].copy()
single['only_lens'] = single['lenses_topk']
single_out = single[['stop_id', 'stop_name', 'region', 'lat', 'lon', 'only_lens',
                     'n_lenses_evaluated', 'consensus_mean_pct', 'reasons']]
single_out = single_out.sort_values(['only_lens', 'consensus_mean_pct'],
                                    ascending=[True, False])
single_out.to_csv(TABLES / 'lens_specific_stations.csv', index=False, encoding='utf-8-sig')

per_lens = single['only_lens'].value_counts().reindex(lens_names).fillna(0).astype(int)
print(f'{len(single):,} stations are flagged by exactly one lens while at least one '
      'other lens evaluated them and did not flag them.')

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(per_lens.index.astype(str), per_lens.to_numpy(), color='#dc2626')
for bar, val in zip(bars, per_lens.to_numpy()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f'{val:,}',
            ha='center', va='bottom', fontsize=10)
ax.set_xlabel('lens')
ax.set_ylabel('stations flagged by this lens alone')
ax.set_title('Lens-specific flags: how idiosyncratic is each definition of critical?')
plt.setp(ax.get_xticklabels(), rotation=25, ha='right')
ax.margins(y=0.14)
plt.tight_layout()
plt.savefig(FIGURES / 'lens_specific_counts.png', dpi=FIG_DPI)
plt.show()

single_out.head(15)

## 24. Summary record

A single JSON with the headline numbers, the lenses that were available and the lenses that were not, the mean and range of the pairwise agreement, and the top of the shortlist. Written last so that it always describes the run that just happened - including, importantly, which lenses were missing, so a reader of the JSON alone cannot mistake a four-lens run for an eight-lens one.

In [ ]:
# --- Summary JSON ----------------------------------------------------------
rho_vals = agreement['spearman_rho'].dropna()
top_rows_json = shortlist.head(10)[['shortlist_rank', 'stop_id', 'stop_name',
                                    'n_lenses_topk', 'consensus_mean_pct', 'reasons']]

summary = {
    'stations_in_universe': int(len(spine)),
    'top_k_definition': int(TOP_K),
    'lenses_available': lens_names,
    'lenses_missing': [{'lens': a, 'needs': b, 'missing': c} for a, b, c in MISSING_LENSES],
    'lens_coverage': {n: LENS_META[n]['stations_evaluated'] for n in lens_names},
    'lens_pairs_compared': int(len(agreement)),
    'spearman_rho_mean': round(float(rho_vals.mean()), 4) if len(rho_vals) else None,
    'spearman_rho_min': round(float(rho_vals.min()), 4) if len(rho_vals) else None,
    'spearman_rho_max': round(float(rho_vals.max()), 4) if len(rho_vals) else None,
    'top_k_overlap_mean': round(float(agreement['top50_overlap'].mean()), 2),
    'stations_flagged_by_any_lens': int((prof['n_lenses_topk'] > 0).sum()),
    'stations_flagged_by_two_or_more': int((prof['n_lenses_topk'] >= 2).sum()),
    'stations_flagged_by_three_or_more': int((prof['n_lenses_topk'] >= 3).sum()),
    'stations_flagged_by_exactly_one': int(len(single)),
    'max_lenses_agreeing_on_one_station': int(prof['n_lenses_topk'].max()),
    'shortlist_size': int(len(shortlist)),
    'unreachable_penalty_seconds': int(UNREACHABLE_PENALTY_SECONDS),
    'articulation_severity_exact': bool(AP_SEVERITY_EXACT),
    'shortlist_top10': top_rows_json.to_dict(orient='records'),
}

with open(STAGE / 'lens_synthesis_summary.json', 'w', encoding='utf-8') as fh:
    json.dump(summary, fh, ensure_ascii=False, indent=2)

print(json.dumps({k: v for k, v in summary.items() if k != 'shortlist_top10'},
                 ensure_ascii=False, indent=2))
print('\nfiles written under', STAGE)
for p in sorted(STAGE.rglob('*')):
    if p.is_file():
        print('   ', p.relative_to(STAGE))

## Takeaways

* **"Critical" is not one property, and this notebook is the proof.** Eight definitions were available in principle; each one produces a different top 50, and the Spearman matrix in section 17 shows how far apart they are. The two lenses that come from the same table (betweenness and service volume) agree the most, which is a sanity check rather than a finding; the interesting numbers are the off-diagonal cells connecting lenses built from different data - structure, walking distance, passenger seconds, mode counts.
* **Rank correlation and top-50 overlap tell different stories, and the top-50 overlap is the one that matters.** Two lenses can correlate strongly across 30,000 stations - because they agree that the vast majority of stops are unremarkable - and still share very few names at the very top. Any claim in this project of the form "station X is the most critical in the country" is a claim about one lens, and section 16 quantifies how little that survives a change of definition.
* **A small robust core does exist.** The stations that several unrelated lenses independently place in their top 50 are the defensible answer to the project's research question. They are exported in `critical_station_shortlist.csv` with the reason for each, and they are the only stations in this project supported by more than one kind of evidence.
* **The single-lens flags are mostly artifacts, and naming them is part of the result.** A cut vertex on a dead-end branch, a stop with no walking alternative because nothing else is within a kilometre of it, a "multimodal hub" that is two route types sharing a kerb - each is a real consequence of its lens's definition and not a national vulnerability. `lens_specific_stations.csv` lists them, and the bar chart in section 23 shows which lens is the most idiosyncratic.
* **Honest limits.** (1) Coverage is unequal: the walking-alternative lens only ever saw a pre-filtered ~10% of the network, so its correlations are computed on that subset and its absence from a station means "unmeasured", not "fine". (2) `consensus_mean_pct` is coverage-conditional and is used only as a tie-break, never as a headline score. (3) The betweenness lens is a sampled approximation whose tail is unstable, so small rank differences are noise. (4) The closure-cost lens depends on `UNREACHABLE_PENALTY_SECONDS`, an assumption about what an impossible journey costs, not a measurement. (5) Everything here is schedule-derived: the GTFS feed contains no passenger counts, so "demand" is always a population proxy and every statement about passengers inherits that.
* **What a planner should take from this.** Do not act on any single ranking in this project. Act on the intersection - and where a station is flagged by one lens only, treat that as a question to investigate rather than an answer, because the lens that flagged it is telling you something specific about *why* it might matter.